<a href="https://colab.research.google.com/github/Nayeem1267/California-House-Price-Prediction-Regression-/blob/main/02_titanic_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset("titanic")
print(titanic.shape)
print(titanic.isna().sum())
print(titanic["survived"].value_counts(normalize=True).round(3))
titanic.head()
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data = titanic[["survived","pclass","sex","age","sibsp","parch","fare","embarked"]].copy()
data = data.dropna(subset=["embarked"])          # only 2 rows

X = pd.get_dummies(data.drop(columns="survived"), columns=["sex","embarked"], drop_first=True)
y = data["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# fill missing ages using the TRAINING median only
median_age = X_train["age"].median()
X_train = X_train.assign(age=X_train["age"].fillna(median_age))
X_test  = X_test.assign(age=X_test["age"].fillna(median_age))

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline accuracy:", round(accuracy_score(y_test, dummy.predict(X_test)), 3))

logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("Logistic regression accuracy:", round(accuracy_score(y_test, logreg.predict(X_test)), 3))

print(pd.Series(logreg.coef_[0], index=X.columns).round(2).sort_values())
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

pred = logreg.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_pipe = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000))
rf_pipe = make_pipeline(SimpleImputer(strategy="median"),
                        RandomForestClassifier(n_estimators=200, random_state=42))

for name, p in [("Logistic", lr_pipe), ("Random Forest", rf_pipe)]:
    s = cross_val_score(p, X, y, cv=cv, scoring="accuracy")
    print(name, s.round(3), "mean:", s.mean().round(3))
    X2 = X.copy()
X2["family_size"] = X2["sibsp"] + X2["parch"] + 1
X2["is_child"] = (X2["age"] < 13).astype(int)   # missing ages count as 0 (not child)

for name, p in [("Logistic", lr_pipe), ("Random Forest", rf_pipe)]:
    s = cross_val_score(p, X2, y, cv=cv, scoring="accuracy")
    print(name, s.round(3), "mean:", s.mean().round(3))
    from sklearn.model_selection import RepeatedStratifiedKFold

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for feat_name, data_X in [("original", X), ("with new features", X2)]:
    for name, p in [("Logistic", lr_pipe), ("Random Forest", rf_pipe)]:
        s = cross_val_score(p, data_X, y, cv=rcv, scoring="accuracy", n_jobs=-1)
        print(f"{feat_name:18} {name:14} mean {s.mean():.3f}  std {s.std():.3f}")

(891, 15)
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
survived
0    0.616
1    0.384
Name: proportion, dtype: float64
Baseline accuracy: 0.618
Logistic regression accuracy: 0.815
sex_male     -2.55
pclass       -1.01
embarked_S   -0.31
sibsp        -0.24
parch        -0.09
age          -0.04
fare          0.00
embarked_Q    0.32
dtype: float64
[[98 12]
 [21 47]]
              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.69      0.74        68

    accuracy                           0.81       178
   macro avg       0.81      0.79      0.80       178
weighted avg       0.81      0.81      0.81       178

Logistic [0.792 0.803 0.775 0.803 0.78 ] mean: 0.791
R